# Convergence Analysis: Cultural Sensitivity Index
Unified 3D convergence metric and Kruskal-Wallis tests across cultures.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
from src.analysis.data_loading import load_annotations
from src.analysis.convergence import compute_convergence
sns.set_theme(style='whitegrid')

In [ ]:
df = load_annotations(model_names=['qwen_vl'])
cap_embs = np.load('../outputs/analysis/caption_embs_qwen_vl.npy')
just_embs = np.load('../outputs/analysis/just_embs_qwen_vl.npy')

conv_df = compute_convergence(df, cap_embs, just_embs)
conv_df.to_csv('../outputs/analysis/convergence_all_dimensions.csv', index=False)
print(f'Convergence computed for {len(conv_df)} images')
conv_df.describe().round(4)

In [ ]:
# Cultural Sensitivity Index = mean per-image std of predicted sentiments
pivot_sent = df.pivot_table(index='image_id', columns='culture', values='predicted_sentiment', aggfunc='mean')
csi = pivot_sent.std(axis=1).mean()
print(f'Cultural Sensitivity Index (CSI): {csi:.4f}')
print(f'  (higher = more cultural variation in sentiment perception)')

In [ ]:
# Kruskal-Wallis test: do cultures differ in predicted sentiment?
culture_groups = [grp['predicted_sentiment'].dropna().tolist() for _, grp in df.groupby('culture')]
stat, p_val = kruskal(*culture_groups)
print(f'Kruskal-Wallis H={stat:.3f}, p={p_val:.6f}')
print('Cultures differ significantly in predicted sentiment.' if p_val < 0.05 else 'No significant culture difference.')

In [ ]:
# Boxplots: caption similarity per image grouped by majority ground-truth sentiment
merged = conv_df.merge(df[['image_id','ground_truth_sentiment']].drop_duplicates(), on='image_id')
fig, ax = plt.subplots(figsize=(10, 6))
merged.boxplot(column='caption_sim', by='ground_truth_sentiment', ax=ax)
ax.set_title('Caption Similarity by Ground-Truth Sentiment')
ax.set_xlabel('σ₃P₅ Sentiment (0=neg → 4=pos)')
ax.set_ylabel('Mean cosine similarity')
plt.tight_layout()
plt.savefig('../outputs/figures/05_convergence_boxplots.pdf', bbox_inches='tight')
plt.show()